# Interactive Ekman assimilation
Run from the project root. Install: `pip install -e ".[test,notebook]"`.

Both `ug` and `K` are uncertain; only selected variables are updated. Forecast times are seconds after spin-up in the Python API and hours in the panel. The first setup uses the supplied 72-hour spin-up and can take a few seconds.

In [ ]:
from src import EkmanExperiment, EkmanConfig
exp = EkmanExperiment.from_yaml("configs/config.yaml", EkmanConfig(), seed=14)

## Run, observe, update, repeat
An `ug`-only update retains each member's uncertain `K`. Interior winds respond during the subsequent forecast; the top boundary responds immediately.

In [ ]:
exp.run_until(3600)
obs = exp.observe(heights=[100, 200], variables=("u", "v"), std=0.25)
record = exp.assimilate(obs, variables=("ug",))
exp.run_until(7200)
record2 = exp.assimilate(exp.observe(), variables=("ug", "K"))
exp.run_until(10800)
print("Time:", exp.time, "Analysis steps:", len(exp.analyses))

## Enter real or chosen observations
Ordering: all `u` heights, followed by all `v` heights. Use the same observation object only once in a branch. Reusing a measurement as though it were new would double-count its information.

In [ ]:
manual = exp.observe([7.2, 8.6, 2.0, 1.2], heights=[100, 200], variables=("u", "v"), std=0.5)
# Choose when to apply it:
# exp.assimilate(manual, variables=("ug", "K", "u", "v"))

## Optional control panel
Run this cell in JupyterLab or a notebook frontend supporting ipywidgets. Choose truth/bias/spread and click **Apply & reset** for a new experiment. **Run to time** pauses at the selected absolute time; **Stop run** requests an earlier cooperative stop. Prepare synthetic or entered observations, choose variables under **Update**, and click **Assimilate**. Repeat as often as needed. A pending observation is cleared after a forecast/reset. The panel's initial pause target below is one hour beyond the current time.

In [ ]:
from src.widgets import ExperimentPanel
panel = ExperimentPanel(exp)
panel.until.value = exp.time/3600 + 1
panel.display();

## Edit and save the next experiment
Errors are signed prior-center biases, separate from ensemble spread. These operations require an explicit reset and erase the current run only when `reset()` is called.

In [ ]:
# exp.set_parameters(ug_truth=10, K_truth=2, ug_error=-2, K_error=1,
#                    sigma_ug=1.5, spread_K=0.8, save_to="configs/my_experiment.yaml")
# exp.reset()
# branch = exp.copy()  # independent checkpoint, including RNG state
# exp.datasets()["ensemble"].to_netcdf("ensemble.nc", engine="scipy")